In [16]:
import pandas as pd
import os

# Load the datasets
try:
    train_df = pd.read_csv(r'/data/womens_uk/2.clean\cleaned_train_data.csv', encoding='latin1', on_bad_lines='skip')
    test_df = pd.read_csv(r'/data/womens_uk/2.clean\cleaned_test_data.csv', encoding='latin1', on_bad_lines='skip')

    print("Cleaned training data loaded successfully.")
    print("Cleaned test data loaded successfully.")

except FileNotFoundError as e:
    print(e)
    print("\n Please make sure the files 'cleaned_training_data.csv' and 'cleaned_test_data.csv' are uploaded.")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8840\572772890.py:7: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(r'C:\Users\ADMIN\return_risk\data\womens_uk\clean\cleaned_test_data.csv', encoding='latin1', on_bad_lines='skip')


Cleaned training data loaded successfully.
Cleaned test data loaded successfully.


In [17]:
from collections import Counter
import re

# Combine the 'Name' and 'Description' columns
text_data = ' '.join(train_df['Name'].fillna('') + ' ' + train_df['Description'].fillna(''))

# Tokenize the text data
words = re.findall(r'\w+', text_data.lower())

# Count the frequency of each word
word_counts = Counter(words)

# Display the most common words
print("Most common words:")
display(word_counts.most_common(20))

Most common words:


[('a', 88006),
 ('the', 82570),
 ('and', 79181),
 ('with', 57507),
 ('for', 49386),
 ('to', 41340),
 ('in', 36675),
 ('this', 29954),
 ('of', 29343),
 ('is', 25513),
 ('your', 23999),
 ('fit', 21166),
 ('not', 19545),
 ('available', 19540),
 ('s', 18198),
 ('from', 17887),
 ('dress', 17788),
 ('length', 17638),
 ('style', 16924),
 ('on', 16142)]

In [18]:
print("Training Data Columns:")
print(train_df.columns)

print("\nTest Data Columns:")
print(test_df.columns)

Training Data Columns:
Index(['Product ID', 'Name', 'Retailer', 'Brand', 'Segment', 'Gender',
       'Category', 'Color', 'Activewear', 'Pattern',
       ...
       'trousers.1', 'waist.2', 'white.1', 'wide.1', 'women', 'womens',
       'wool.1', 'woven', 'wrap', 'zip.2'],
      dtype='object', length=323)

Test Data Columns:
Index(['Product ID', 'Name', 'Retailer', 'Brand', 'Segment', 'Gender',
       'Category', 'Color', 'Activewear', 'Pattern',
       ...
       'trousers.1', 'waist.2', 'white.1', 'wide.1', 'women', 'womens',
       'wool.1', 'woven', 'wrap', 'zip.2'],
      dtype='object', length=323)


In [19]:
from collections import Counter
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

# Identify common words
text_data = ' '.join(train_df['Name'].fillna('') + ' ' + train_df['Description'].fillna(''))
words = re.findall(r'\w+', text_data.lower())
stop_words = set(stopwords.words('english'))
words = [word for word in words if not word in stop_words]
word_counts = Counter(words)
print("Most common words (after removing stop words):")
display(word_counts.most_common(20))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Most common words (after removing stop words):


[('fit', 21166),
 ('available', 19540),
 ('dress', 17788),
 ('length', 17638),
 ('style', 16924),
 ('size', 15126),
 ('design', 14085),
 ('neck', 13183),
 ('perfect', 12326),
 ('top', 12322),
 ('fabric', 11322),
 ('fastening', 11030),
 ('made', 10360),
 ('long', 10049),
 ('5', 9994),
 ('front', 9926),
 ('look', 9639),
 ('sleeves', 9410),
 ('sleeve', 9355),
 ('0', 9204)]

In [20]:
import numpy as np

def assign_return_risk(row):
    # High Risk
    if row['Category'] in ['Dresses', 'Jumpsuits & Playsuits'] and (
        'sequin' in str(row['Name']).lower() or 'beaded' in str(row['Name']).lower() or 'lace' in str(row['Name']).lower() or 'bodycon' in str(row['Name']).lower() or 'plunge' in str(row['Name']).lower() or 'backless' in str(row['Name']).lower()
    ):
        return 'High'
    if row['Full Price ($)'] > 200:
        return 'High'

    # Medium Risk
    if row['Category'] in ['Tops', 'Bottoms', 'Knitwear']:
        return 'Medium'
    if row['Current Discount Percentage'] > 20:
        return 'Medium'

    # Low Risk
    if row['Category'] in ['Accessories', 'Shoes', 'Bags']:
        return 'Low'
    if 'cotton' in str(row['Care information']).lower():
        return 'Low'

    return 'Low'  # Default to low risk

# Apply the logic to the training and test data
train_df['return_risk'] = train_df.apply(assign_return_risk, axis=1)
test_df['return_risk'] = test_df.apply(assign_return_risk, axis=1)

# Display the distribution of return risk
print("Return Risk Distribution in Training Data:")
display(train_df['return_risk'].value_counts())

print("\nReturn Risk Distribution in Test Data:")
display(test_df['return_risk'].value_counts())

Return Risk Distribution in Training Data:


return_risk
High      28212
Medium    24373
Low       14791
Name: count, dtype: int64


Return Risk Distribution in Test Data:


return_risk
High      29936
Low       22641
Medium    15922
Name: count, dtype: int64

In [22]:
# Define the directory
output_directory = r'C:\Users\ADMIN\return_risk\data\womens_uk\with_risk'

# Ensure the directory exists
os.makedirs(output_directory, exist_ok=True)

# Define the full file paths, including filenames
output_train_filepath = os.path.join(output_directory, 'risk_train_data.csv')
output_test_filepath = os.path.join(output_directory, 'risk_test_data.csv')

# Save the cleaned DataFrames to CSV files
train_df.to_csv(output_train_filepath, index=False)
test_df.to_csv(output_test_filepath, index=False)

print(f"Predicted risk training data saved to: {output_train_filepath}")
print(f"Predicted risk test data saved to: {output_test_filepath}")

Predicted risk training data saved to: C:\Users\ADMIN\return_risk\data\womens_uk\with_risk\risk_train_data.csv
Predicted risk test data saved to: C:\Users\ADMIN\return_risk\data\womens_uk\with_risk\risk_test_data.csv
